# UniMedia — Python quickstart

`unimedia` is a Cython extension over the UniMedia C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install lituus-unimedia
```

## The API

In [1]:
import unimedia

unimedia.engine_version(), unimedia.__version__, unimedia.abi_version()

('1.0.0', '1.0.0', 1)

## Answers a path alone can give

Most of the library reads files, but a useful part of it answers from the name
before anything is opened. A domain holds some kinds of file and not others: a
photo library has no category for a video, and says so with an empty string
rather than an error.

In [2]:
print("photo / b.jpg =", repr(unimedia.category_for("photo", "a/b.jpg")))
print("photo / b.mp4 =", repr(unimedia.category_for("photo", "a/b.mp4")))
print("video / b.mp4 =", repr(unimedia.category_for("video", "a/b.mp4")))

try:
    unimedia.category_for("sculpture", "a/b.jpg")
except ValueError as exc:
    print("unknown domain ->", exc)

photo / b.jpg = 'image'
photo / b.mp4 = ''
video / b.mp4 = 'video'
unknown domain -> domain is one of music, photo, video, visual


## What a name claims about its date

`filename_date` reads the **date** a name claims, not the time: a name carrying
`101500` still answers at midnight. A name claiming nothing answers nothing,
which is not an error.

In [3]:
print("IMG_20240115_101500.jpg ->", repr(unimedia.filename_date("IMG_20240115_101500.jpg")))
print("holiday.jpg             ->", repr(unimedia.filename_date("holiday.jpg")))

IMG_20240115_101500.jpg -> '2024-01-15 00:00:00'
holiday.jpg             -> ''


## What can be written, without reading anything

Whether a date correction can reach a file, or metadata be stripped from it,
follows from the format. Answering from the name costs no read.

In [4]:
for name in ("a.jpg", "notes.txt"):
    print(f"{name:10} date-writable={unimedia.can_write_date(name)}"
          f" strippable={unimedia.can_strip(name)}")

a.jpg      date-writable=True strippable=True
notes.txt  date-writable=False strippable=False


## A path that escapes its root is refused

`checked_path_under` resolves a path and refuses it when it leaves the root —
the guard between a library and the rest of the disk. `is_internal` separates
the library's own bookkeeping from the media it holds.

In [5]:
import os
import tempfile

root = tempfile.mkdtemp()
inside = os.path.join(root, "one.ppm")

print("inside  ->", os.path.basename(unimedia.checked_path_under(root, inside)))
try:
    unimedia.checked_path_under(root, os.path.join(root, "..", "elsewhere.jpg"))
except unimedia.UniMediaError as exc:
    print("outside ->", type(exc).__name__)

print("the database is internal   =",
      unimedia.is_internal(root, os.path.join(root, ".organizeMedia.db")))
print("the photo next to it is not =",
      unimedia.is_internal(root, inside))

inside  -> one.ppm
outside -> UniMediaError
the database is internal   = True
the photo next to it is not = False


## Reading a file

A PPM is a plain-text image, which makes it a fair subject without shipping a
binary: two pixels by two, solid red. `probe_still` reports what the bytes say,
`blake3_file` digests them, and `perceptual_hash_file` reduces the picture to a
hash that survives re-encoding.

In [6]:
with open(inside, "w") as handle:
    handle.write("P3\n2 2\n255\n" + "255 0 0\n" * 4)

print("probe_still =", unimedia.probe_still(inside))
print("blake3      =", unimedia.blake3_file(inside)[:16], "...",
      len(unimedia.blake3_file(inside)), "hex characters")
print("perceptual  =", unimedia.perceptual_hash_file(inside))

probe_still = {'width': 2, 'height': 2, 'format': 'ppm', 'codec': 'ppm'}
blake3      = 6fd3eb9c2027277d ... 64 hex characters
perceptual  = {'hash': '1', 'width': 2, 'height': 2}


## Coordinates the file did not carry

A file with no location does not fail and does not guess: the answer says it
was not found, and the numbers alongside are not an estimate.

In [7]:
print(unimedia.media_coordinates(inside))

{'latitude': 0.0, 'longitude': 0.0, 'found': False}


## An AppleDouble sidecar

macOS leaves `._name` files behind. Whether one can be deleted depends on what
it holds, and the verdict carries the reason — here the bytes do not parse as
AppleDouble at all, so the honest answer is that its contents are unknown.

In [8]:
sidecar = os.path.join(root, "._photo.jpg")
with open(sidecar, "wb") as handle:
    handle.write(b"\x00\x05\x16\x07")

verdict = unimedia.apple_double_verdict(sidecar)
print("removable =", verdict["removable"])
print("reason    =", verdict["reason"])

removable = False
reason    = not readable as AppleDouble, so what it holds is unknown


## A GPX track

`parse_gpx` returns the track points as timestamps and coordinates, which is
what dating a photo from a track needs.

In [9]:
track = os.path.join(root, "walk.gpx")
with open(track, "w") as handle:
    handle.write(
        '<?xml version="1.0"?><gpx><trk><trkseg>'
        '<trkpt lat="43.6" lon="1.44"><time>2024-01-15T10:15:00Z</time></trkpt>'
        '<trkpt lat="43.7" lon="1.45"><time>2024-01-15T10:20:00Z</time></trkpt>'
        '</trkseg></trk></gpx>')

points = unimedia.parse_gpx(track)
print("points =", len(points))
for point in points:
    print("  ", point)

points = 2
   {'timestamp': 1705313700, 'latitude': 43.6, 'longitude': 1.44}
   {'timestamp': 1705314000, 'latitude': 43.7, 'longitude': 1.45}


## Sync manifests

A manifest lists what a side holds. `diff_sync_manifests` compares two of them
without touching either side's files; a manifest against itself differs in
nothing.

In [10]:
import json

manifest = json.dumps({"schemaVersion": 1, "entries": [
    {"path": "a.jpg", "digest": "ab" * 32, "size": 1, "mtimeNs": 2}]})
empty = json.dumps({"schemaVersion": 1, "entries": []})

print("entry        =", unimedia.parse_sync_manifest(manifest)["entries"][0]["path"])
print("against self =", unimedia.diff_sync_manifests(manifest, manifest))
print("against none =", unimedia.diff_sync_manifests(manifest, empty))

entry        = a.jpg
against self = {'onlyLocal': [], 'onlyRemote': [], 'changed': []}
against none = {'onlyLocal': ['a.jpg'], 'onlyRemote': [], 'changed': []}


## A digest that is not a digest

The manifest is validated, not trusted: an entry whose digest is not 64 hex
characters is refused rather than carried forward.

In [11]:
try:
    unimedia.parse_sync_manifest(
        '{"schemaVersion":1,"entries":[{"path":"a","digest":"ab","size":1,"mtimeNs":2}]}')
except unimedia.UniMediaError as exc:
    print("refused ->", type(exc).__name__)

refused -> UniMediaError


## Identifiers and timestamps

`iso_now` stamps an instant and `new_batch_id` names a run. Neither value can be
printed here — they differ every time, which is the point of them — so what is
shown is their shape.

In [12]:
stamp = unimedia.iso_now()

print("iso_now length   =", len(stamp), "| ends with Z =", stamp.endswith("Z"))
print("two batch ids differ =", unimedia.new_batch_id() != unimedia.new_batch_id())

iso_now length   = 20 | ends with Z = True
two batch ids differ = True


## Configuration

`default_config` is the starting point a caller edits, and reading it back
tells you the keys that exist rather than guessing at them.

In [13]:
print(sorted(unimedia.default_config()))

['birthtimeDate', 'domain', 'filenameDate', 'noDateDir', 'onConflict', 'schemaVersion', 'scheme']
